# NLP — โจทย์แบบ "สร้าง/แปลงข้อความ" (Text Generation / Seq2Seq)

ข้อความเข้า -> ข้อความออก เช่น แปลภาษา, สรุปความ, ตอบคำถามจากบทความ, แก้คำผิด, ถอดความ

วิธีในโน้ตบุ๊กนี้:
- วิธีที่ 1 (ง่ายสุด) = `pipeline` ของ HuggingFace ใช้โมเดลสำเร็จรูป generate เลย ไม่ต้องเทรน
- วิธีที่ 2 (ไม่บังคับ ต้อง GPU) = fine-tune `mT5` ด้วย Seq2SeqTrainer


## เมื่อไหร่ใช้โน้ตบุ๊กนี้

ใช้เมื่อ output เป็น "ข้อความใหม่" (ไม่ใช่ป้าย/คลาส) วัดด้วย BLEU/ROUGE/accuracy แล้วแต่งาน
ถ้า output เป็นป้าย -> ใช้ `text_classification.ipynb` หรือ `thai_word_segmentation.ipynb`

ต้องแก้ (`# << แก้`): ชื่อ competition, `SRC_COL` (ข้อความเข้า), `TGT_COL` (ข้อความเป้าหมาย), `TASK`/โมเดล

## ขั้นที่ 1 — ติดตั้ง

In [ ]:
!pip -q install -U transformers sentencepiece sacremoses pandas torch
!pip -q install datasets evaluate sacrebleu rouge_score   # วิธีที่ 2 (เทรน/วัดผล)

## ขั้นที่ 2 — ตั้งค่า Kaggle แล้วดาวน์โหลดข้อมูล

ต้องมีไฟล์ `kaggle.json` ก่อน วิธีได้มา: เข้า kaggle.com -> กดรูปโปรไฟล์ -> Settings -> Account -> Create New Token
จะได้ไฟล์ `kaggle.json` หน้าตา `{"username":"...","key":"..."}`

- ถ้ารันบน `Kaggle Notebook`: ข้อมูลอยู่ที่ `/kaggle/input/...` แล้ว ไม่ต้องทำอะไร รันผ่านได้เลย
- ถ้ารันบน `Google Colab / เครื่องตัวเอง`: เอา username กับ key ใส่ในเซลล์ข้างล่าง (จุด `# << แก้`)

In [ ]:
import os, glob, subprocess

COMP = "ใส่-slug-ของ-competition-textgen-ตรงนี้"      # << แก้ตรงนี้: slug ของ competition (คือส่วนท้าย URL หลังคำว่า /competitions/)
KAGGLE_USERNAME = ""   # << แก้ (เฉพาะ Colab/เครื่องตัวเอง) เช่น "peetwan1"
KAGGLE_KEY      = ""   # << แก้ (เฉพาะ Colab/เครื่องตัวเอง) คีย์ยาว ๆ จาก kaggle.json

def get_data(comp):
    # ถ้าอยู่บน Kaggle ข้อมูลถูกต่อไว้ให้แล้ว
    kpath = f"/kaggle/input/{comp}"
    if os.path.isdir(kpath):
        print("อยู่บน Kaggle แล้ว ข้อมูลอยู่ที่", kpath); return kpath
    # ถ้าไม่ใช่ Kaggle: เขียน token แล้วโหลดด้วย kaggle CLI
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        open(os.path.expanduser("~/.kaggle/kaggle.json"), "w").write(
            '{"username":"%s","key":"%s"}' % (KAGGLE_USERNAME, KAGGLE_KEY))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    os.makedirs("data", exist_ok=True)
    subprocess.run(["kaggle","competitions","download","-c",comp,"-p","data"], check=True)
    for z in glob.glob("data/*.zip"):
        subprocess.run(["unzip","-o","-q",z,"-d","data"], check=True)
    return "data"

DATA_DIR = get_data(COMP)
print("ไฟล์ที่ดาวน์โหลดมา (ดูชื่อไฟล์/โฟลเดอร์ แล้วเอาไปแก้ในเซลล์ถัดไป):")
for p in sorted(glob.glob(os.path.join(DATA_DIR, "**"), recursive=True))[:40]:
    print("  ", p)

## ขั้นที่ 3 — โหลดข้อมูล + CONFIG

In [ ]:
import os, pandas as pd
TRAIN_CSV=os.path.join(DATA_DIR,"train.csv"); TEST_CSV=os.path.join(DATA_DIR,"test.csv")
SAMPLE_SUB=os.path.join(DATA_DIR,"sample_submission.csv")
train=pd.read_csv(TRAIN_CSV); test=pd.read_csv(TEST_CSV); sample=pd.read_csv(SAMPLE_SUB)
SRC_COL="source"   # << แก้: คอลัมน์ข้อความเข้า (เช่น ประโยคต้นทาง/บทความ)
TGT_COL=sample.columns[1]   # << แก้ถ้าผิด: คอลัมน์ข้อความเป้าหมาย
print("train คอลัมน์:",list(train.columns)); display(train.head()); display(sample.head())

## วิธีที่ 1 — pipeline สำเร็จรูป (ง่ายสุด ไม่ต้องเทรน)

เลือก TASK + โมเดลให้ตรงงาน:
- สรุปความไทย: model="csebuetnlp/mT5_multilingual_XLSum", task="summarization"
- แปล ไทย<->อังกฤษ: model="Helsinki-NLP/opus-mt-th-en" (หรือ -en-th), task="translation"
- ทั่วไป seq2seq: task="text2text-generation" 

In [ ]:
from transformers import pipeline
import torch
TASK  = "summarization"                         # << แก้: summarization / translation / text2text-generation
MODEL = "csebuetnlp/mT5_multilingual_XLSum"     # << แก้โมเดลให้ตรงงาน/ภาษา
gen = pipeline(TASK, model=MODEL, device=0 if torch.cuda.is_available() else -1)
outs=[]
for txt in test[SRC_COL].astype(str).tolist():
    r = gen(txt, max_length=128, truncation=True)[0]   # << แก้ max_length ตามความยาวเป้าหมาย
    outs.append(r.get("summary_text") or r.get("translation_text") or r.get("generated_text"))
out=sample.copy(); out[TGT_COL]=outs
out.to_csv("submission.csv",index=False,encoding="utf-8-sig")
print("บันทึก submission.csv"); display(out.head())

## วิธีที่ 2 — fine-tune mT5 (ไม่บังคับ ต้อง GPU)

เทรน seq2seq บนคู่ (source -> target) ของเราเอง

In [ ]:
import numpy as np, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer)
MODEL="google/mt5-small"   # << แก้: mt5-small เร็ว / mt5-base แม่นกว่า
tok=AutoTokenizer.from_pretrained(MODEL)
model=AutoModelForSeq2SeqLM.from_pretrained(MODEL)
ds=Dataset.from_dict({"src":train[SRC_COL].astype(str).tolist(),
                      "tgt":train[TGT_COL].astype(str).tolist()}).train_test_split(test_size=0.1,seed=42)
def prep(b):
    enc=tok(b["src"], max_length=256, truncation=True)
    lab=tok(text_target=b["tgt"], max_length=128, truncation=True)
    enc["labels"]=lab["input_ids"]; return enc
ds=ds.map(prep, batched=True, remove_columns=ds["train"].column_names)
# หมายเหตุ: ถ้า transformers เก่า (ก่อน 4.46) เปลี่ยน eval_strategy -> evaluation_strategy
args=Seq2SeqTrainingArguments(output_dir="out", learning_rate=3e-4, per_device_train_batch_size=8,
    num_train_epochs=3, eval_strategy="epoch", save_strategy="no",
    predict_with_generate=True, fp16=torch.cuda.is_available(), report_to="none")
trainer=Seq2SeqTrainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["test"],
    tokenizer=tok, data_collator=DataCollatorForSeq2Seq(tok, model=model))
trainer.train()
model.eval(); outs=[]
for txt in test[SRC_COL].astype(str).tolist():
    ids=tok(txt, return_tensors="pt", truncation=True, max_length=256).to(model.device)
    with torch.no_grad(): g=model.generate(**ids, max_new_tokens=128, num_beams=4)
    outs.append(tok.decode(g[0], skip_special_tokens=True))
out=sample.copy(); out[TGT_COL]=outs; out.to_csv("submission_mt5.csv",index=False,encoding="utf-8-sig")
print("บันทึก submission_mt5.csv")

## ขั้นสุดท้าย — ส่ง submission ขึ้น Kaggle

เช็คก่อนส่งทุกครั้ง: ไฟล์ submission ต้องมีชื่อคอลัมน์ + จำนวนแถว ตรงกับ `sample_submission.csv` เป๊ะ ๆ
- บน Kaggle Notebook: กดปุ่ม `Submit` ที่แถบขวา (ง่ายสุด) หรือใช้คำสั่งข้างล่าง
- บน Colab/เครื่อง: รันเซลล์ข้างล่าง (ต้องตั้ง token แล้ว)

In [ ]:
FILE = "submission.csv"   # << แก้เป็นชื่อไฟล์ที่คะแนนดีสุด
!kaggle competitions submit -c {COMP} -f {FILE} -m "pipeline text generation"
!kaggle competitions submissions -c {COMP} | head